Teste do segundo dataset, e com a nova função de treinamento

In [1]:
from pathlib import Path
import sys
import time

sys.path.insert(0, str(Path.cwd().resolve().parent.parent))

from ultralytics import YOLO
import torch
import src.config as cfg
from src.training_logger import TrainingLogger

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

NVIDIA GeForce RTX 4060 Ti


In [2]:
ROOT = Path.cwd().resolve().parent.parent
PRETRAINED_DIR = ROOT / "pretrained_models"

logger = TrainingLogger(log_dir=str(ROOT / "training_logs"))

MODEL_SIZE = "s"
EPOCHS = 100
BATCH_SIZE = 16
IMGSZ = 768

In [4]:
model = YOLO(str(PRETRAINED_DIR / f"yolo26{MODEL_SIZE}.pt"))

start = time.time()

results = model.train(
    data=str(cfg.DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    patience=20,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
    close_mosaic=10,
    device=0 if torch.cuda.is_available() else "cpu",
)

train_time = time.time() - start

New https://pypi.org/project/ultralytics/8.4.84 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.75  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\pedro\OneDrive\Documentos\programao\eletiva-leomar\YOLOCraft\data\minecraft_mobs-2\apresentacao\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det

In [5]:
metrics = model.val(workers=0)

baseline_map50, baseline_map50_95 = 0.9262, 0.7663

print(f"mAP50:    {metrics.box.map50:.4f}  ({metrics.box.map50 - baseline_map50:+.4f})")
print(f"mAP50-95: {metrics.box.map:.4f}  ({metrics.box.map - baseline_map50_95:+.4f})")

Ultralytics 8.4.75  Python-3.12.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
YOLO26s summary (fused): 122 layers, 9,471,372 parameters, 0 gradients, 20.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 224.158.4 MB/s, size: 26.8 KB)
val: Scanning C:\Users\pedro\OneDrive\Documentos\programação\eletiva-leomar\YOLOCraft\data\minecraft_mobs-2\apresentacao\val\labels.cache... 479 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 479/479  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 6.1it/s 4.9s0.1s
                   all        479        479      0.988      0.987      0.993      0.968
           cave_spider         39         39      0.989          1      0.995      0.981
               creeper         26         26      0.962      0.974      0.994      0.979
              enderman         32         32      0.968      0.943      0.992      0.917
              skeleton         24        

In [6]:
train_id, _ = logger.log_training(
    model_size=MODEL_SIZE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    imgsz=IMGSZ,
    map50=metrics.box.map50,
    map50_95=metrics.box.map,
    train_time=train_time,
    notes="Dataset 2",
)

logger.summary()


ID                 Model        Epochs   mAP50      mAP50-95   Time(h)  Date
--------------------------------------------------------------------------------
20260702_153623    yolo26s      100      0.9935     0.9679     4.85     02/07/2026
20260630_184356    yolo26s      100      0.9522     0.8165     2.16     30/06/2026

Best: yolo26s (ID: 20260702_153623) — mAP50: 0.9935
